## ML Training

**Lets start with a story**

Suppose shell has `50,000` incident reports. Some are labeled

```text
- Leak

- Gas Detection

- Fire

- Slip/Fall

- Corrosion

- Equipment Failure
```
then; 

Management asks

> Can AI automatically classify every new incident report?

You should immediately answer `Yes`. But then;

`How?`

---

### This brings `Machine Learning`

Definition

> Machine Learning is the process of enabling a computer to discover patterns from historical data in order to make predictions on new, unseen data.

---

`ML flow`;

Historical reports

↓

Model learns

↓

New report

↓

Prediction

---

**Real example**

Historical

```text
Pressure leak detected...

↓

Leak
```

Historical

```text
Worker slipped...

↓

Slip/Fall
```

Historical

```text
Gas detector triggered alarm

↓

Gas Detection
```

`then after learning;`

New report

```text
Gas leakage detected near separator.
```

Model predicts

```text
Gas Detection
```

Nobody wrote

```python
if "gas" in report:
```

The model learned it during training.

---

---

### 1. Supervised Learning

Definition

Students already know the definition and examples of what a `Supervised Learning` is; if you still don't remember visit: https://www.github.com/3Logy....  *to read more.*

---

**Why `split data`?**

Can we train and test using the same data?

If you say `Yes` then you're `wrong`.

`Analogy`:

A teacher gives students exam questions before the exam. Everyone scores 100%.

Did they learn?

`No`.

They memorized. That's exactly the same thing that happens in ML.

---

`Train/Test split`


```text
    Dataset

    ↓

    80%

    Training

    ↓

    20%

    Testing
```

---

Use

```python
from sklearn.model_selection import train_test_split
```

---

Code

```python
X = df["report_text"]

y = df["incident_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Why random_state? For `reproducibility`.
The cell below provide the steps to follow.

---

### Import setup

In [1]:
'''Imports setup'''
#!/usr/bin/env python3
import sys
import logging
# from sklearn.model_selection import train_test_split
from src.data_loader import DataLoader
from src.feature_engineering import FeatureEngineer
# Import the splitter
from src.data_splitter import DataSplitter
from src.model import IncidentClassifier

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

# Automatically finds the project root 'sira' and adds it to Python's path
PROJECT_ROOT = r"c:\Users\M.faisal\Desktop\sira"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

### Load Data

In [2]:
loader = DataLoader()
df = loader.load_data()
df.head(2)

INFO: Loading dataset...
INFO: 
Data Loaded Succesfully...


,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
0,1,pressure leak detected on pipeline 7 during ro...,Platform B,Aisha,Instrumentation,High,Leak,2025-09-30,Day,Open
1,4,smoke observed from electrical control panel,Refinery,Joy,Production,High,Electrical,2026-01-22,Day,Open


### What is `data leakage`?


Should `TF-IDF` or `bow` learn using all data? As you may have guessed the answer is `No`. But `why`?

`Because` the test data must remain unseen. 


`Professional pipeline`

```text
    Training Data

    ↓

    Fit TF-IDF/BOW

    ↓

    Transform Training

    ↓

    Transform Testing
```

**NOT**

```text
    Entire Dataset

    ↓

    Fit TF-IDF

    ↓

    Split
```

This is one of the biggest mistakes beginners make. Be careful

---

### Let's split our dataset using the `data_splitter.py` module:
Create `data_splitter.py` module and add:
```python
"""Data splitting module for SIRA."""

import logging

import pandas as pd
from sklearn.model_selection import train_test_split


class DataSplitter:
    """
    Responsible for splitting input features (X) and target (y)
    into training, validation, and test datasets.

    This class does not perform:
        - Data loading
        - Data cleaning
        - Feature engineering
        - Model training

    It only handles dataset splitting.
    """

    def __init__(
        self,
        test_size: float = 0.2,
        validation_size: float = 0.1,
        random_state: int = 42,
    ):
        """
        Initialize the DataSplitter.

        Parameters
        ----------
        test_size : float
            Proportion of the complete dataset reserved for testing.

        validation_size : float
            Proportion of the complete dataset reserved for validation.

        random_state : int
            Controls reproducibility of the split.
        """

        self.test_size = test_size
        self.validation_size = validation_size
        self.random_state = random_state

        # Validate split configuration
        if test_size <= 0 or test_size >= 1:
            raise ValueError(
                "test_size must be between 0 and 1."
            )

        if validation_size <= 0 or validation_size >= 1:
            raise ValueError(
                "validation_size must be between 0 and 1."
            )

        if test_size + validation_size >= 1:
            raise ValueError(
                "test_size + validation_size must be less than 1."
            )

    def split(
        self,
        X: pd.Series,
        y: pd.Series,
    ):
        """
        Split features and target into train, validation, and test sets.

        Parameters
        ----------
        X : pd.Series
            Input feature data.

        y : pd.Series
            Target labels.

        Returns
        -------
        tuple
            X_train, X_val, X_test, y_train, y_val, y_test
        """

        # Validate inputs
        if not isinstance(X, pd.Series):
            raise TypeError(
                "X must be a pandas Series."
            )

        if not isinstance(y, pd.Series):
            raise TypeError(
                "y must be a pandas Series."
            )

        if len(X) != len(y):
            raise ValueError(
                "X and y must contain the same number of samples."
            )

        if X.empty or y.empty:
            raise ValueError(
                "X and y cannot be empty."
            )

        # First split: Training + Validation/Test
        X_train, X_temp, y_train, y_temp = train_test_split(
            X,
            y,
            test_size=(
                self.test_size + self.validation_size
            ),
            random_state=self.random_state,
            stratify=y,
        )

        # --------------------------------------------------
        # Calculate validation proportion
        #
        # Example:
        #
        # Total = 1000
        # Train = 700
        # Validation = 100
        # Test = 200
        #
        # Remaining data = 300
        #
        # Validation proportion of remaining:
        #
        # 100 / 300 = 0.3333
        # --------------------------------------------------
        validation_ratio = (
            self.validation_size
            / (
                self.test_size
                + self.validation_size
            )
        )

        # Second split: Validation + Test
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp,
            y_temp,
            test_size=(
                1 - validation_ratio
            ),
            random_state=self.random_state,
            stratify=y_temp,
        )

        # Logging
        logging.info("Dataset splitting completed.")

        logging.info(
            "Training samples: %d",
            len(X_train)
        )

        logging.info(
            "Validation samples: %d",
            len(X_val)
        )

        logging.info(
            "Test samples: %d",
            len(X_test)
        )

        return (
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            y_test,
        )
   
```

In [3]:
#Define X and y
X = df["report_text"]
y = df["incident_type"]

#Instantiate the object
splitter = DataSplitter(
    test_size=0.2,
    validation_size=0.1,
    random_state=42
)

In [4]:
(
    X_TRAIN_TEXT,
    X_VAL_TEXT,
    X_TEST_TEXT,
    y_train,
    y_val,
    y_test,
) = splitter.split(
    X=X,
    y=y
)

INFO: Dataset splitting completed.
INFO: Training samples: 646
INFO: Validation samples: 92
INFO: Test samples: 186


In [5]:
#Print shape
print(X_TRAIN_TEXT.shape)
print(X_VAL_TEXT.shape)
print(X_TEST_TEXT.shape)

(646,)
(92,)
(186,)


### Let's now use the `FeatureEngineering` module to engineer our data.

Make sure your module is updated with the script below:
```python
"""Feature engineering module for SIRA."""

import logging
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


class FeatureEngineer:
    """
    Responsible for converting incident report text
    into numerical features.

    Supports:
        - TF-IDF
        - Bag of Words
    """

    def __init__(
        self,
        method: str = "tfidf",
        max_features: int = 5000,
        ngram_range=(1, 1)
    ):
        self.method = method
        self.max_features = max_features
        self.ngram_range = ngram_range

        self.vectorizer = None

        if method == "tfidf":
            self.vectorizer = TfidfVectorizer(
                max_features=max_features,
                ngram_range=ngram_range
            )

        elif method == "bow":
            self.vectorizer = CountVectorizer(
                max_features=max_features,
                ngram_range=ngram_range
            )

        else:
            raise ValueError(
                "method must be either 'tfidf' or 'bow'."
            )

    def fit_transform(
        self,
        text_data: pd.Series,
    ):
        """
        Learn the vocabulary/statistics from training data
        and transform the training data into numerical features.
        """

        if text_data.empty:
            raise ValueError("Text data cannot be empty.")

        logging.info(
            "Fitting %s vectorizer on training data...",
            self.method,
        )

        features = self.vectorizer.fit_transform(text_data)

        logging.info(
            "Feature engineering completed. Shape: %s",
            features.shape,
        )

        return features

    def transform(
        self,
        text_data: pd.Series,
    ):
        """
        Transform validation/test/new data using the
        vectorizer already fitted on training data.
        """

        if self.vectorizer is None:
            raise RuntimeError(
                "FeatureEngineer must be fitted before "
                "calling transform()."
            )

        features = self.vectorizer.transform(text_data)

        logging.info(
            "Data transformed. Shape: %s",
            features.shape,
        )

        return features
    
    # FEATURE NAMES
    def get_feature_names(self):
        """
        Return the vocabulary/features learned
        by the vectorizer.
        """

        return self.vectorizer.get_feature_names_out()

    # VOCABULARY SIZE
    def get_vocabulary_size(self):
        """
        Return the number of features learned.
        """

        return len(
            self.vectorizer.get_feature_names_out()
        )
  
```

In [6]:
feature_engineer = FeatureEngineer(
    method="bow", # Note that we're using Bag of Words (BoW) here, but you can switch to TF-IDF if needed
    max_features=5000,
    ngram_range=(1, 2)
)

In [7]:
X_train = feature_engineer.fit_transform(
    X_TRAIN_TEXT
)

X_val = feature_engineer.transform(
    X_VAL_TEXT
)

X_test = feature_engineer.transform(
    X_TEST_TEXT
)

INFO: Fitting bow vectorizer on training data...
INFO: Feature engineering completed. Shape: (646, 150)
INFO: Data transformed. Shape: (92, 150)
INFO: Data transformed. Shape: (186, 150)


> Let's check the `dimensions` of our train and test dataset.

In [8]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_val.shape)
print("Testing shape:", X_test.shape)

Training shape: (646, 150)
Testing shape: (92, 150)
Testing shape: (186, 150)


Remember that:

`fit_transform()` means: 
> **Learn the representation and transform the data.**

Use primarily on ```text Training data ```.

While:

`transform()`

Means:

> **Use the representation already learned to transform new data.**

Use on:

```text
- Validation data
- Test data
- Production data
- New incident reports
```

**One important** point about the other columns is that, we don't want to blindly feed all 10 columns into the NLP pipeline. For the current SIRA classification task, our deliberate selection is:
```text
X = df["report_text"]
y = df["incident_type"]
```

The remaining columns aren't necessarily useless. They may become useful later. For example, if the dataset eventually contains:
```text
report_text
incident_type
location
severity
timestamp
reporter
...
```

we could later build a `multimodal/tabular + NLP` feature pipeline using some of those fields. But for our current objective: 
>*Classify an incident based on its textual report*; 

we should keep the first ML pipeline focused on 
```text 
report_text → incident_type
```

This will make it much easier for you to understand why each engineering component exists, rather than throwing the whole DataFrame into every stage.

---

### Building the first `ML Pipeline`

Now let's update the project.

```
src/

model.py
```

---

Why another `module`? 

Again `single responsibility.`

---

Inside `model.py` add:

```python
# model.py
import logging
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report)

logging.basicConfig(level=logging.INFO)

class IncidentClassifier:
    """A simple wrapper around scikit-learn's LogisticRegression for classifying incidents."""
    def __init__(self, max_iter=1000, **kwargs):

        self.model = LogisticRegression(max_iter=max_iter, **kwargs)

    def train(self, x_train, y_train):
        """Train the Logistic Regression model on the provided training data."""
        logging.info("Training model...")
        self.model.fit(x_train, y_train)
        
        logging.info("Model training completed.")

    def predict(self, x):
        """Make predictions on the provided data."""
        return self.model.predict(x)
    
    def evaluate(self, X, y):
        """Evaluate model performance."""

        predictions = self.predict(X)

        accuracy = accuracy_score(
            y,
            predictions,
        )

        report = classification_report(
            y,
            predictions,
        )

        return {
            "accuracy": accuracy,
            "classification_report": report,
        }
        
    
```

**Notice:** The `model` knows nothing about `CSV` cleaning, `EDA`, or `streamlit`. Just `one job`.

---

### Why did we choose `Logistic Regression`?

Why not `Deep Learning`?

That's because professional engineers always start with a simple `baseline`. Read about `baseline` here: https://www.github.com/3Logy...

Baseline

↓

Measure performance

↓

Improve.

Never start with `BERT` (an algorithms used to train NLP models which we'll introduce later).

---

### Update `main.py` module

`Workflow`:

```python
    loader = DataLoader()

    processor = TextPreprocessor()

    splitter = DataSplitter(
        test_size=0.2,
        validation_size=0.1,
        random_state=42,
    )

    engineer = FeatureEngineer(
        method="tfidf",
        max_features=5000,
    )

    classifier = IncidentClassifier()
```

`The Pipeline`:

```text
    Load Data

    ↓

    Clean Text

    ↓

    Train/Test Split

    ↓

    TF-IDF

    ↓

    Train Model

    ↓

    Evaluate
```

---

Later we'll use a train.py module to train i.e:
```python

# """Training pipeline for the Smart Incident Report Analyzer."""

# import logging

# from src.data_loader import DataLoader
# from src.preprocessing import TextPreprocessor
# from src.data_splitter import DataSplitter
# from src.feature_engineering import FeatureEngineer
# from src.model import IncidentClassifier


# logger = logging.getLogger(__name__)


# def train_pipeline():
#     """
#     Execute the complete machine learning training pipeline.

#     Returns
#     -------
#     dict
#         Trained model components and evaluation data.
#     """

#     logger.info("Starting training pipeline...")

#     # 1. Initialize components
#     loader = DataLoader()

#     processor = TextPreprocessor()

#     splitter = DataSplitter(
#         test_size=0.2,
#         validation_size=0.1,
#         random_state=42,
#     )

#     engineer = FeatureEngineer(
#         method="tfidf",
#         max_features=5000,
#     )

#     classifier = IncidentClassifier()

#     # 2. Load dataset
#     logger.info("Loading dataset...")

#     df = loader.load_data()

#     logger.info(
#         "Raw data shape: %s",
#         df.shape,
#     )

#     # 3. Preprocess dataset
#     logger.info("Preprocessing dataset...")

#     processed_df = processor.preprocess_dataset(df)

#     logger.info(
#         "Processed data shape: %s",
#         processed_df.shape,
#     )

#     # 4. Split data
#     logger.info("Splitting dataset...")

#     (
#         x_train,
#         x_val,
#         x_test,
#         y_train,
#         y_val,
#         y_test,
#     ) = splitter.split(
#         processed_df["report_text"],
#         processed_df["incident_type"],
#     )

#     # 5. Feature engineering
#     logger.info("Engineering features...")

#     x_train_features = engineer.fit_transform(
#         x_train
#     )

#     x_val_features = engineer.transform(
#         x_val
#     )

#     x_test_features = engineer.transform(
#         x_test
#     )

#     logger.info(
#         "Training features: %s",
#         x_train_features.shape,
#     )

#     logger.info(
#         "Validation features: %s",
#         x_val_features.shape,
#     )

#     logger.info(
#         "Test features: %s",
#         x_test_features.shape,
#     )

#     # 6. Train model
#     logger.info("Training model...")

#     classifier.train(
#         x_train_features,
#         y_train,
#     )

#     logger.info("Training completed.")

#     # 7. Return trained components
#     return {
#         "classifier": classifier,
#         "engineer": engineer,
#         "processor": processor,
#         "x_val_features": x_val_features,
#         "y_val": y_val,
#         "x_test_features": x_test_features,
#         "y_test": y_test,
#     }


# if __name__ == "__main__":
#     logging.basicConfig(
#         level=logging.INFO,
#         format="%(levelname)s: %(message)s",
#     )

#     train_pipeline()
    
```

### `Training`

Let's start training the model

In [9]:
classifier = IncidentClassifier()

In [11]:
#Training
classifier.train(
    X_train,
    y_train
)

INFO: Training model...
INFO: Model training completed.


### `Prediction`

In [ ]:
predictions = classifier.predict(
    X_test
)

In [19]:
# print(predictions[0:6])
print(predictions)

['Oil Spill' 'Equipment Failure' 'Inspection' 'Equipment Failure'
 'Slip/Fall' 'Maintenance' 'Mechanical' 'Leak' 'Inspection' 'Inspection'
 'Maintenance' 'Electrical' 'Inspection' 'Inspection' 'Gas Detection'
 'Maintenance' 'Maintenance' 'Equipment Failure' 'Oil Spill' 'Electrical'
 'Corrosion' 'Equipment Failure' 'Electrical' 'Electrical' 'Gas Detection'
 'Corrosion' 'Leak' 'Gas Detection' 'Inspection' 'Inspection'
 'Equipment Failure' 'Equipment Failure' 'Equipment Failure' 'Security'
 'Leak' 'Leak' 'Gas Detection' 'Equipment Failure' 'Security'
 'Equipment Failure' 'Slip/Fall' 'Maintenance' 'Inspection'
 'Gas Detection' 'Gas Detection' 'Equipment Failure' 'Mechanical'
 'Gas Detection' 'Corrosion' 'Mechanical' 'Equipment Failure' 'Slip/Fall'
 'Inspection' 'Leak' 'Inspection' 'Equipment Failure' 'Slip/Fall' 'Leak'
 'Oil Spill' 'Security' 'Equipment Failure' 'Oil Spill'
 'Equipment Failure' 'Mechanical' 'Corrosion' 'Electrical' 'Inspection'
 'Corrosion' 'Maintenance' 'Slip/Fall' 'Leak'

> You now have your first NLP AI model built from scratch.

### `Evaluation`

Question to ask yourself. `How good is the model`? i.e `Accuracy`?

Not enough.

**Create/update `evaluate.py` with the script below:

```python


"""Model evaluation and benchmarking."""

import logging

from train import train_pipeline


logger = logging.getLogger(__name__)


def evaluate_model(
    classifier,
    features,
    labels,
    dataset_name,
):
    """
    Evaluate a trained classifier on a dataset.

    Parameters
    ----------
    classifier : IncidentClassifier
        Trained classifier.

    features : array-like
        Feature matrix.

    labels : array-like
        True labels.

    dataset_name : str
        Name of dataset being evaluated.

    Returns
    -------
    dict
        Evaluation metrics.
    """

    results = classifier.evaluate(
        features,
        labels,
    )

    logger.info(
        "%s Accuracy: %.4f",
        dataset_name,
        results["accuracy"],
    )

    return results


def evaluate_pipeline():
    """Train and evaluate the current model."""

    logger.info("Starting evaluation pipeline...")

    pipeline = train_pipeline()

    classifier = pipeline["classifier"]

    # Validation
    validation_results = evaluate_model(
        classifier=classifier,
        features=pipeline["x_val_features"],
        labels=pipeline["y_val"],
        dataset_name="Validation",
    )

    # Test
    test_results = evaluate_model(
        classifier=classifier,
        features=pipeline["x_test_features"],
        labels=pipeline["y_test"],
        dataset_name="Test",
    )

    logger.info(
        "Evaluation pipeline completed."
    )

    return {
        "validation": validation_results,
        "test": test_results,
    }


if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(levelname)s: %(message)s",
    )

    evaluate_pipeline()
    
```

In [20]:
from sklearn.metrics import accuracy_score

accuracy_score(
    y_test,
    predictions
)

1.0

Lets not celebrate the **1.0000 validation and test accuracy** yet. With a real-world NLP classification problem, getting:

```text
Validation Accuracy: 1.0000
Test Accuracy: 1.0000
```

on only **924 records** is suspiciously perfect. It doesn't necessarily mean our model is wrong, but it strongly suggests we should check for **data leakage** or an overly easy/synthetic dataset. For example, the dataset may contain wording that directly reveals  incident_type`.

Imagine:

```text
incident_type: Fire
report_text: "A fire incident occurred in the storage facility..."
```

versus:

```text
incident_type: Theft
report_text: "A theft incident was reported..."
```

A TF-IDF classifier can essentially learn:

```text
"fire"  → Fire
"theft" → Theft
"injury" → Injury
"equipment failure" → Equipment Failure
```

That can produce extremely high accuracy without demonstrating that the model can generalize to genuinely unseen incident reports.

---

### `confusion matrix`

If you don't know what a `confusion matrix` is? Visit: https://www.github.com/3Logy...

---

You should have a visualization

```
                Predicted

                Leak Fire

    Actual Leak  40    3

    Actual Fire   2   55
```

Now this immediately gives you the understanding of `mistakes`.

### `classification report`

If you don't remember what a `classification report` is, Visit: https://www.github.com/3Logy...

In [21]:
from sklearn.metrics import classification_report

print(
    classification_report(
    y_test,
    predictions
))

                   precision    recall  f1-score   support

        Corrosion       1.00      1.00      1.00        11
       Electrical       1.00      1.00      1.00        15
Equipment Failure       1.00      1.00      1.00        40
    Gas Detection       1.00      1.00      1.00        12
       Inspection       1.00      1.00      1.00        24
             Leak       1.00      1.00      1.00        27
      Maintenance       1.00      1.00      1.00        11
       Mechanical       1.00      1.00      1.00        13
        Oil Spill       1.00      1.00      1.00        12
         Security       1.00      1.00      1.00         9
        Slip/Fall       1.00      1.00      1.00        12

         accuracy                           1.00       186
        macro avg       1.00      1.00      1.00       186
     weighted avg       1.00      1.00      1.00       186



Our current pipeline has achieved:

```text
incident_reports_1000.csv
        │
        ▼
   DataLoader
        │
        ▼
  924 records
        │
        ▼
 TextPreprocessor
        │
        ▼
 Cleaned dataset
        │
        ▼
   DataSplitter
   ┌────┼─────┐
   ▼    ▼     ▼
 Train Val   Test
 646   92    186
   │    │     │
   ▼    ▼     ▼
       TF-IDF
   fit     transform
   │         │
   ▼         ▼
  Features (70)
        │
        ▼
   IncidentClassifier
        │
        ├── Validation → 100%
        │
        └── Test → 100%
```

### We can also train the model using the `main.py` module:

```python
"""Application entry point."""

from train import train_pipeline
from evaluate import evaluate_pipeline
from predict import predict_incident

def main():
    """Run the SIRA machine learning pipeline."""
    
    # 1. Train the pipeline ONCE and store the outputs
    pipeline_artifacts = train_pipeline()
    
    # 2. Extract components safely (assumes train_pipeline returns a dict)
    classifier = pipeline_artifacts.get("classifier")
    engineer = pipeline_artifacts.get("engineer")
    processor = pipeline_artifacts.get("processor")

    #  `evaluate_pipeline`` handles its own loading
    evaluate_pipeline()
    
    # 4. Predict using the already trained artifacts
    print("Running incident prediction...")
    result = predict_incident(
        report_text="A pipeline leak was detected in the northern section of the facility, causing a temporary shutdown of operations.",
        classifier=classifier,
        engineer=engineer,
        processor=processor,
    )
    
    print(f"Predicted incident type: {result}")



if __name__ == "__main__":
    main()
    
```

### Saving the `model`.

Remember that professional engineers never retrain every time. The load from a saved model.

```
models/
```

Use:

In [ ]:
import joblib


# Save
joblib.dump(
classifier.model,
"models/incident_classifier.pkl"
)

['models/incident_classifier.pkl']

**Save TF-IDF too**.

People usually forget this. Prediction requires the `same vocabulary`.

In [23]:
joblib.dump(
    feature_engineer.vectorizer,
    "models/vectorizer.pkl"
)

['models/vectorizer.pkl']

**Load `model`**: If you plan to make predictions in the future, you should not train a new model but use the trained model.

In [24]:
model = joblib.load(
    "models/incident_classifier.pkl"
)

To predict **`new incident`**:

```python
    new_report = [
    "Gas leakage detected near separator."
    ]
```

*Transform*

```python
    features = vectorizer.transform(
    new_report
    )
```

*Predict*

```python
    prediction = model.predict(
    features
    )
```

*Output*

```
Gas Detection
```

You now have your first working **AI** application.

---

### Updated project structure

```
    smart_incident_report_analyzer/

    │
    ├── data/
    │
    ├── models/
    │   ├── incident_classifier.pkl
    │   └── vectorizer.pkl
    │
    ├── notebooks/
    |   └── ***
    │
    ├── src/
    │   ├── __init__.py
    │   ├── incident.py
    │   ├── data_loader.py
    │   ├── eda.py
    │   ├── preprocessing.py
    |   ├── data_splitter.py
    |   ├── clean_data.py
    │   ├── feature_engineering.py
    │   ├── model.py
    │   └── utils.py
    │
    ├── train.py
    ├── predict.py
    ├── main.py
    ├── requirements.txt
    ├── README.md
    └── .gitignore
```

---

**New files introduced**

1. `train.py`

Responsible for:

* Loading the dataset.
* Cleaning and preprocessing text.
* Splitting the data.
* Training the vectorizer.
* Training the classifier.
* Evaluating the model.
* Saving the trained artifacts.

This script is run only when you want to create or update the model.

Add:
```python
"""Training pipeline for the Smart Incident Report Analyzer."""

import logging

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor
from src.data_splitter import DataSplitter
from src.feature_engineering import FeatureEngineer
from src.model import IncidentClassifier


logger = logging.getLogger(__name__)


def train_pipeline():
    """
    Execute the complete machine learning training pipeline.

    Returns
    -------
    dict
        Trained model components and evaluation data.
    """

    logger.info("Starting training pipeline...")

    # 1. Initialize components
    loader = DataLoader()

    processor = TextPreprocessor()

    splitter = DataSplitter(
        test_size=0.2,
        validation_size=0.1,
        random_state=42,
    )

    engineer = FeatureEngineer(
        method="tfidf",
        max_features=5000,
    )

    classifier = IncidentClassifier()

    # 2. Load dataset
    logger.info("Loading dataset...")

    df = loader.load_data()

    logger.info(
        "Raw data shape: %s",
        df.shape,
    )

    # 3. Preprocess dataset
    logger.info("Preprocessing dataset...")

    processed_df = processor.preprocess_dataset(df)

    logger.info(
        "Processed data shape: %s",
        processed_df.shape,
    )

    # 4. Split data
    logger.info("Splitting dataset...")

    (
        x_train,
        x_val,
        x_test,
        y_train,
        y_val,
        y_test,
    ) = splitter.split(
        processed_df["report_text"],
        processed_df["incident_type"],
    )

    # 5. Feature engineering
    logger.info("Engineering features...")

    x_train_features = engineer.fit_transform(
        x_train
    )

    x_val_features = engineer.transform(
        x_val
    )

    x_test_features = engineer.transform(
        x_test
    )

    logger.info(
        "Training features: %s",
        x_train_features.shape,
    )

    logger.info(
        "Validation features: %s",
        x_val_features.shape,
    )

    logger.info(
        "Test features: %s",
        x_test_features.shape,
    )

    # 6. Train model
    logger.info("Training model...")

    classifier.train(
        x_train_features,
        y_train,
    )

    logger.info("Training completed.")

    # 7. Return trained components
    return {
        "classifier": classifier,
        "engineer": engineer,
        "processor": processor,
        "x_val_features": x_val_features,
        "y_val": y_val,
        "x_test_features": x_test_features,
        "y_test": y_test,
    }


if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(levelname)s: %(message)s",
    )

    train_pipeline()
    
```

---

2. `predict.py`

Responsible for:

* Loading the saved model.
* Loading the saved TF-IDF vectorizer.
* Accepting new incident reports.
* Applying the same preprocessing pipeline.
* Generating predictions.

Separating training from prediction reflects how production ML systems are built.

Add:

```python
"""Prediction pipeline for new incident reports."""

import logging
import pandas as pd

logger = logging.getLogger(__name__)


def predict_incident(
    report_text,
    classifier,
    engineer,
    processor,
):
    """
    Predict the incident type for a new report.

    Parameters
    ----------
    report_text : str
        New incident report.

    classifier : IncidentClassifier
        Trained classifier.

    engineer : FeatureEngineer
        Fitted feature engineer.

    processor : TextPreprocessor
        Text preprocessing component.

    Returns
    -------
    str
        Predicted incident type.
    """

    logger.info(
        "Processing new incident report..."
    )

    # preprocess_dataset() expects the same structural columns 
    # that existed during training. Since these values are not 
    # supplied for a new report, we use safe default values.
    new_data = pd.DataFrame(
        {
            "report_text": [report_text],
            "location": ["Unknown"], 
            "reported_by": ["Unknown"], 
            "department": ["Unknown"], 
            "severity": ["Unknown"], 
            "status": ["Unknown"], 
            "shift": ["Unknown"], 
            "report_date": [pd.NaT],
        }
    )

    # 2. Preprocess
    processed_data = processor.preprocess_dataset(
        new_data
    )

    # 3. Transform using fitted vectorizer
    features = engineer.transform(
        processed_data["report_text"]
    )

    # 4. Predict
    prediction = classifier.predict(
        features
    )

    return prediction[0]

```

3. `model.py` and `data_splitter.py`

---

**End of `phase 4_01` deliverables**

By the end of this phase, students should have:

* A clean and modular ML project structure.
* A reusable `IncidentClassifier` class.
* A `train.py` script that trains and evaluates the model.
* A `predict.py` script that performs inference on new reports.
* Saved model artifacts (`incident_classifier.pkl` and `vectorizer.pkl`).
* A complete understanding of:

  * Train/test splitting
  * Data leakage
  * TF-IDF fitting vs. transforming
  * Baseline models
  * Model evaluation (Accuracy, Precision, Recall, F1-score)
  * Model persistence with `joblib`

---

---

**`Recommended`** task.

Although your trained model uses **Logistic Regression** as the baseline, don't stop here. This allows you to compare multiple classical algorithms on the same dataset:

* **Logistic Regression** (baseline)
* **Naive Bayes** (excellent for text classification)
* **Linear Support Vector Machine (LinearSVC)** (often a top performer for TF-IDF features)
* **Random Forest** (to illustrate why tree-based models are usually less effective on sparse text vectors)
* **Decision Tree** (for comparison and interpretability)

Train with the above listed models, then compare:

| Model | Accuracy | Precision | Recall | F1-score | Training Time |
| ----- | -------- | --------- | ------ | -------- | ------------- |

This exercise reinforces an important engineering principle: **don't assume a model is best; rather measure it.** That mindset is central to both professional ML engineering and the AWS Machine Learning certification path.